In [2]:
import javalang
import os

def analyze_java_code(code):
    """
    Analyze structure of Java code using javalang parser
    """
    tree = javalang.parse.parse(code)
    info = {
        "classes": [],
        "methods": [],
        "imports": [],
    }

    # Get class and method names
    for _, node in tree:
        if isinstance(node, javalang.tree.ClassDeclaration):
            info["classes"].append(node.name)
        elif isinstance(node, javalang.tree.MethodDeclaration):
            info["methods"].append(node.name)
        elif isinstance(node, javalang.tree.Import):
            info["imports"].append(node.path)

    return info


if __name__ == "__main__":
    # Example Java code
    java_code = """
    import java.util.*;

    public class Calculator {
        public int add(int a, int b) {
            return a + b;
        }
        public void unusedMethod() {
            // nothing
        }
    }
    """

    analysis = analyze_java_code(java_code)
    print("Analysis:", analysis)


Analysis: {'classes': ['Calculator'], 'methods': ['add', 'unusedMethod'], 'imports': ['java.util']}


In [4]:
from openai import OpenAI

client = OpenAI()

def ai_code_review(java_code, structure):
    """
    Sends structured info + code to GPT for intelligent review
    """
    prompt = f"""
    You are an expert Java reviewer.
    Here is code structure:
    Classes: {structure['classes']}
    Methods: {structure['methods']}
    Imports: {structure['imports']}

    Review this Java code for:
    - Code smells
    - Redundant logic
    - Unused methods/imports
    - Better design patterns

    Code:
    {java_code}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a senior Java code reviewer."},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content


if __name__ == "__main__":
    java_code = open("data/Calculator.java").read()  # or paste code here
    structure = analyze_java_code(java_code)
    review = ai_code_review(java_code, structure)
    print("\n=== AI Code Review ===\n")
    print(review)



=== AI Code Review ===

Here is a detailed review of the provided Java code for the `Calculator` class:

### Code Smells
1. **Inconsistent Naming**: 
   - The method `Subtract` should be renamed to `subtract` to maintain consistent naming conventions in Java, which typically use lower camel case for method names.

2. **Unnecessary Side Effects**:
   - The `multiply` method has a side effect of printing to the console, which can create confusion. It's a better practice to keep business logic separate from I/O operations.

3. **Poor Exception Handling**:
   - The `divide` method prints a message instead of throwing an exception for a division by zero error. Throwing a dedicated exception (e.g., `ArithmeticException`) would provide a clearer indication of an error.

4. **Unclear Handling of Edge Cases**:
   - The `sqrt` method only returns 0 for input 0 but does not handle negative inputs, which can lead to unexpected results (NaN). A better strategy would be to throw an exception for ne